In [1]:
include("src/HybridPropagation.jl")

In [2]:
#Evaluation
eps = 1e-12

function evaluate(psum::PauliSum, hsum::HybridSum) 
    length(psum) == length(hsum) ? print("Equal amount of terms\n") : print("NON equal amount of terms\n")
    for (term, coeff) in psum.terms
        try 
            diff = abs(hsum.terms[term] - coeff)
            diff > eps ? print("Significant difference of $(diff) $(bitstring(term))) \n") : nothing
        catch e
            print("term not part of the Hybrid sum \n")
        end 
    end 
    return
end

function evaluate(msum::MajoranaSum, hsum::HybridSum) 
    length(msum) == length(hsum) ? print("Equal amount of terms\n") : print("NON equal amount of terms\n")
    for (term, coeff) in msum.Majoranas
        try 
            diff = abs(hsum.terms[term] - coeff)
            diff > eps ? print("Significant difference of $(diff) (for $(bitstring(term))) \n") : nothing
        catch e
            print("term not part of the Hybrid sum \n")
        end 
    end 
    return
end

evaluate (generic function with 2 methods)

# Pauli Propagation only:

In [3]:
#Initialise Pauli sum 
nq = 6 
pstr = PauliString(nq, [:X, :Y, :Z], [1, 3, 5])
psum = PauliSum(pstr)

PauliSum(nqubits: 6, 1 Pauli term: 
 1.0 * XIYIZI
)

In [4]:
#Initialise Hybrid sum
hsum = HybridSum(pstr)

HybridSum with 1 term:(
    1.0 *  x XIYIZI) 


In [5]:
#Construct gates
gate_symb1 = [:Z, :Z]
gate_ind1 = [1, 2]
gate_symb2 = [:X, :X]
gate_ind2 = [2, 4]
gate_symb3 = [:Z, :Z]
gate_ind3 = [4, 5]

#Pauli gates
pcirc = Gate[]
push!(pcirc, PauliRotation(gate_symb1, gate_ind1))
push!(pcirc, PauliRotation(gate_symb2, gate_ind2))
push!(pcirc, PauliRotation(gate_symb3, gate_ind3))

#Hybrid gates
hcirc = Gate[]
push!(hcirc, HybridGate(0, nq, gate_symb1, gate_ind1))
push!(hcirc, HybridGate(0, nq, gate_symb2, gate_ind2))
push!(hcirc, HybridGate(0, nq, gate_symb3, gate_ind3))

theta = - 1.0 * 0.1
thetas = [theta*0.5, theta, theta*1.5] 

3-element Vector{Float64}:
 -0.05
 -0.1
 -0.15000000000000002

### First Step

In [6]:
#For Pauli Propagation 
propagate!(pcirc, psum, thetas .* 2)

PauliSum(nqubits: 6, 2 Pauli terms:
 0.099833 * YZYIZI
 0.995 * XIYIZI
)

In [7]:
#For Hybrid Propagation
f_filter, q_filter = create_filters(0, nq, false)
propagate!(hcirc, hsum, thetas, f_filter, q_filter)

HybridSum with 2 terms:(
    0.09983341664682815 *  x YZYIZI
    0.9950041652780258 *  x XIYIZI) 


In [8]:
evaluate(psum, hsum)

Equal amount of terms


### Second Step

In [9]:

#For Pauli Propagation 
propagate!(pcirc, psum, thetas .* 2)

PauliSum(nqubits: 6, 3 Pauli terms:
 0.19669 * YZYIZI
 -0.019834 * YYYXZI
 0.98027 * XIYIZI
)

In [10]:
#For Hybrid Propagation
propagate!(hcirc, hsum, thetas, f_filter, q_filter)

HybridSum with 3 terms:(
    0.19668925097469325 *  x YZYIZI
    -0.019833838076209875 *  x YYYXZI
    0.9802652485007214 *  x XIYIZI) 


In [11]:
evaluate(psum, hsum)

Equal amount of terms


### Third Step

In [12]:
#For Pauli Propagation 
propagate!(pcirc, psum, thetas .* 2)

PauliSum(nqubits: 6, 4 Pauli terms:
 -0.0058613 * YYYYII
 0.28592 * YZYIZI
 -0.057646 * YYYXZI
 0.9565 * XIYIZI
)

In [13]:
#For Hybrid Propagation
propagate!(hcirc, hsum, thetas, f_filter, q_filter)

HybridSum with 4 terms:(
    -0.005861299927169088 *  x YYYYII
    0.28592317210909446 *  x YZYIZI
    -0.05764641283088305 *  x YYYXZI
    0.9564990726090479 *  x XIYIZI) 


In [14]:
evaluate(psum, hsum)

Equal amount of terms


### Check Truncations:

In [15]:
min_coeff = 1e-2
print("\nFirst Coefficient Truncation\n")
for i in 1:3
    propagate!(pcirc, psum, thetas .* 2, min_abs_coeff=min_coeff)
    propagate!(hcirc, hsum, thetas, f_filter, q_filter, min_abs_coeff=min_coeff)
    print("Round $(i): \n")
    evaluate(psum, hsum)
    print(hsum, "\n")
end 


First Coefficient Truncation
Round 1: 
Equal amount of terms
HybridSum with 4 terms:(
    -0.022635193527201343 *  x YYYYII
    0.3637703638342623 *  x YZYIZI
    -0.10908051379359328 *  x YYYXZI
    0.9248027962330135 *  x XIYIZI) 

Round 2: 
Equal amount of terms
HybridSum with 4 terms:(
    -0.053859722293962964 *  x YYYYII
    0.42778695601882777 *  x YZYIZI
    -0.16784555663008915 *  x YYYXZI
    0.8865242916548304 *  x XIYIZI) 

Round 3: 
Equal amount of terms
HybridSum with 4 terms:(
    -0.10105591158410623 *  x YYYYII
    0.477118968694065 *  x YZYIZI
    -0.22654146582190934 *  x YYYXZI
    0.8431038825135836 *  x XIYIZI) 



In [16]:
max_pauli_weight = 4
print("\nNow Pauli Weight Truncation:\n")
for i in 1:3
    propagate!(pcirc, psum, thetas .* 2, max_weight=max_pauli_weight)
    propagate!(hcirc, hsum, thetas, f_filter, q_filter, max_pauli_weight=max_pauli_weight)
    print("Round $(i): \n")
    evaluate(psum, hsum)
    print(hsum, "\n")
end 



Now Pauli Weight Truncation:
Round 1: 
Equal amount of terms
HybridSum with 3 terms:(
    -0.16348998057520114 *  x YYYYII
    0.5494422019951214 *  x YZYIZI
    0.7922089351437032 *  x XIYIZI) 

Round 2: 
Equal amount of terms
HybridSum with 3 terms:(
    -0.15618794404992617 *  x YYYYII
    0.6148886565916102 *  x YZYIZI
    0.734491899834989 *  x XIYIZI) 

Round 3: 
Equal amount of terms
HybridSum with 3 terms:(
    -0.14921204211240305 *  x YYYYII
    0.6729480083010724 *  x YZYIZI
    0.6706597059864108 *  x XIYIZI) 



# For Majorana Propagation only

In [17]:
#Initialise Majorana sum 
nf = 3
msum = MajoranaSum(nf, :n, 1) 

MajoranaSum with 2 terms:
    0.5 * 00000000
    0.5 * 11000000

In [18]:
#Initialise Hybrid sum
hsum = HybridSum(msum)

HybridSum with 2 terms:(
    0.5 * 000000 x 
    0.5 * 110000 x ) 


In [19]:
#Construct gates
gate_symb123 = :hop
gate_ind1 = [1, 2]
gate_ind2 = [1, 3]
gate_ind3 = [2,3]
gate_symb456 = :n
gate_ind4 = [1]
gate_ind5 = [2]
gate_ind6 = [3]

#Majorana gates
mcirc = Gate[]
push!(mcirc, FermionicGate(gate_symb123, gate_ind1))
push!(mcirc, FermionicGate(gate_symb123, gate_ind2))
push!(mcirc, FermionicGate(gate_symb123, gate_ind3))
push!(mcirc, FermionicGate(gate_symb456, gate_ind4))
push!(mcirc, FermionicGate(gate_symb456, gate_ind5))
push!(mcirc, FermionicGate(gate_symb456, gate_ind6))

#Hybrid gates
hcirc = Gate[]
push!(hcirc, HybridGate(nf, gate_symb123, gate_ind1, 0))
push!(hcirc, HybridGate(nf, gate_symb123, gate_ind2, 0))
push!(hcirc, HybridGate(nf, gate_symb123, gate_ind3, 0))
push!(hcirc, HybridGate(nf, gate_symb456, gate_ind4, 0))
push!(hcirc, HybridGate(nf, gate_symb456, gate_ind5, 0))
push!(hcirc, HybridGate(nf, gate_symb456, gate_ind6, 0))

theta123 = - 0.5 * 0.1 
theta456 = - 1.0 * 0.1
thetas = [theta123, theta123, theta123, theta456, theta456, theta456]

6-element Vector{Float64}:
 -0.05
 -0.05
 -0.05
 -0.1
 -0.1
 -0.1

### First Step

In [20]:
#Majorana Propagation 
propagate!(mcirc, msum, thetas)

MajoranaSum with 10 terms:
    0.5 * 00000000
    -0.024927162718034697 * 01000100
    -0.02489601025554485 * 10100000
    0.0012473978073654951 * 00100100
    -0.02489601025554485 * 01010000
    0.0012458388849223984 * 00110000
    -0.0012473978073654951 * 00011000
    -0.024927162718034697 * 10001000
    0.4975052024345841 * 11000000
    0.0012489586804935585 * 00001100

In [21]:
#Hybrid Propagation 
f_filter, q_filter = create_filters(nf, 0, false)
propagate!(hcirc, hsum, thetas, f_filter, q_filter)

HybridSum with 10 terms:(
    0.5 * 000000 x 
    -0.024927162718034697 * 010001 x 
    -0.02489601025554485 * 101000 x 
    0.0012473978073654951 * 001001 x 
    -0.02489601025554485 * 010100 x 
    0.0012458388849223984 * 001100 x 
    -0.0012473978073654951 * 000110 x 
    -0.024927162718034697 * 100010 x 
    ...) 


In [22]:
evaluate(msum, hsum)

Equal amount of terms


### Second Step

In [23]:
#Majorana Propagation 
propagate!(mcirc, msum, thetas)

MajoranaSum with 16 terms:
    -0.04923341469946424 * 10100000
    1.5534092506960253e-7 * 00101000
    0.00494917794540807 * 00110000
    -0.004961579312161462 * 00011000
    -0.0493568200256903 * 10001000
    0.5 * 00000000
    0.00124427412747411 * 01100000
    0.004961579312161461 * 00100100
    -0.00124427412747411 * 10010000
    0.49007681029616673 * 11000000
    -0.0012458466584476698 * 10000100
    1.5534092506961608e-7 * 00010100
    0.004974011758425401 * 00001100
    -0.04935682002569029 * 01000100
    -0.04923341469946424 * 01010000
    0.0012458466584476698 * 01001000

In [24]:
#Hybrid Propagation 
propagate!(hcirc, hsum, thetas, f_filter, q_filter)

HybridSum with 16 terms:(
    -0.04923341469946424 * 101000 x 
    1.5534092506960253e-7 * 001010 x 
    0.00494917794540807 * 001100 x 
    -0.004961579312161462 * 000110 x 
    -0.0493568200256903 * 100010 x 
    0.5 * 000000 x 
    0.00124427412747411 * 011000 x 
    0.004961579312161461 * 001001 x 
    ...) 


In [25]:
evaluate(msum, hsum)

Equal amount of terms


### Third step:

In [26]:
propagate!(mcirc, msum, thetas)

MajoranaSum with 16 terms:
    -0.07246661038792639 * 10100000
    1.3887643603760806e-6 * 00101000
    0.01101765732647944 * 00110000
    -0.011059137805718183 * 00011000
    -0.07273990758435009 * 10001000
    0.5 * 00000000
    0.0037048263422114016 * 01100000
    0.01105913780571818 * 00100100
    -0.0037048263422114025 * 10010000
    0.4778815680432745 * 11000000
    -0.0037096403353113496 * 10000100
    1.3887643603758095e-6 * 00010100
    0.011100774630246422 * 00001100
    -0.07273990758435007 * 01000100
    -0.0724666103879264 * 01010000
    0.003709640335311348 * 01001000

In [27]:
propagate!(hcirc, hsum, thetas, f_filter, q_filter)

HybridSum with 16 terms:(
    -0.07246661038792639 * 101000 x 
    1.3887643603760806e-6 * 001010 x 
    0.01101765732647944 * 001100 x 
    -0.011059137805718183 * 000110 x 
    -0.07273990758435009 * 100010 x 
    0.5 * 000000 x 
    0.0037048263422114016 * 011000 x 
    0.01105913780571818 * 001001 x 
    ...) 


In [28]:
evaluate(msum, hsum)

Equal amount of terms


### Check Truncation Schemes

In [29]:
min_coeff = 1e-3
print("\nFirst Coefficient Truncation\n")
for i in 1:3
    propagate!(mcirc, msum, thetas, min_abs_coeff=min_coeff)
    propagate!(hcirc, hsum, thetas, f_filter, q_filter, min_abs_coeff=min_coeff)
    print("Round $(i): \n")
    evaluate(msum, hsum)
    print(length(hsum), "\n")
end 


First Coefficient Truncation
Round 1: 
Equal amount of terms
14
Round 2: 
Equal amount of terms
14
Round 3: 
Equal amount of terms
14


In [30]:
max_unpaired = 1
print("\n Now Unpaired Truncation:\n")
for i in 1:3
    propagate!(mcirc, msum, thetas, max_unpaired=max_unpaired)
    propagate!(hcirc, hsum, thetas, f_filter, q_filter, max_unpaired=max_unpaired)
    print("Round $(i): \n")
    evaluate(msum, hsum)
    print(length(hsum), "\n")
end


 Now Unpaired Truncation:
Round 1: 
Equal amount of terms
4
Round 2: 
Equal amount of terms
4
Round 3: 
Equal amount of terms
4


In [31]:
max_majorana_weight = 1
print("\nFinally Majorana Weight Truncation:\n")
for i in 1:3
    propagate!(mcirc, msum, thetas, max_weight=max_majorana_weight)
    propagate!(hcirc, hsum, thetas, f_filter, q_filter, max_majorana_weight=max_majorana_weight)
    print("Round $(i): \n")
    evaluate(msum, hsum)
    print(length(hsum), "\n")
end


Finally Majorana Weight Truncation:
Round 1: 
Equal amount of terms
1
Round 2: 
Equal amount of terms
1
Round 3: 
Equal amount of terms
1


# Hybrid Propagation

In [32]:
pstr = PauliString(nq, [:X, :Y, :Z], [1, 3, 5])
msum = MajoranaSum(nf, :n, 1)
hsum = HybridSum(msum, pstr)

HybridSum with 2 terms:(
    0.5 * 110000 x XIYIZI
    0.5 * 000000 x XIYIZI) 


In [33]:
gate_symb1_p = [:Z, :Z]
gate_ind1_p = [1, 2]
gate_symb2_p = [:X, :X]
gate_ind2_p = [2, 4]

gate_symb1_m = :hop
gate_ind1_m = [1, 3]
gate_symb2_m = :n
gate_ind2_m = [3]

circ = Gate[]
push!(circ, HybridGate(nf, gate_symb1_m, gate_ind1_m, nq, gate_symb1_p, gate_ind1_p, false))
push!(circ, HybridGate(nf, gate_symb2_m, gate_ind2_m, nq, gate_symb2_p, gate_ind2_p, false))

theta = - 1.0 * 0.1
thetas = [theta, theta]

2-element Vector{Float64}:
 -0.1
 -0.1

In [34]:

f_f, q_f = create_filters(hsum)
propagate!(circ, hsum, thetas, f_f, q_f)

HybridSum with 5 terms:(
    0.5 * 110000 x XIYIZI
    -0.04966733269876531 * 010010 x YZYIZI
    0.49501664446031046 * 000000 x XIYIZI
    -0.004983355539689593 * 110011 x XIYIZI
    0.04966733269876531 * 100001 x YZYIZI) 


In [35]:
propagate!(circ, hsum, thetas, f_f, q_f)

HybridSum with 5 terms:(
    0.5 * 110000 x XIYIZI
    -0.09735458557716264 * 010010 x YZYIZI
    0.48026524850072133 * 000000 x XIYIZI
    -0.019734751499278735 * 110011 x XIYIZI
    0.09735458557716264 * 100001 x YZYIZI) 


In [36]:
propagate!(circ, hsum, thetas, f_f, q_f)

HybridSum with 5 terms:(
    0.5 * 110000 x XIYIZI
    -0.14116061834875887 * 010010 x YZYIZI
    0.45633390372741967 * 000000 x XIYIZI
    -0.04366609627258044 * 110011 x XIYIZI
    0.14116061834875887 * 100001 x YZYIZI) 
